# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library in Python.

### Dataset Source
This dataset is published as a [Croissant](https://mlcommons.github.io/croissant/) schema at:

```text
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

It contains human protein data from mass spectrometry analysis following extracellular vesicle (EV) isolation from mast cells.

In [ ]:
# Ensure that `mlcroissant` is available and install if missing
%pip install mlcroissant pandas --quiet

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`. This will validate the Croissant schema and prepare access to its record sets.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"Dataset name: {dataset.metadata.name}")
print(f"Dataset description: {dataset.metadata.description}")
print(f"Published on: {getattr(dataset.metadata, 'datePublished', None)}")
print(f"Keywords: {getattr(dataset.metadata, 'keywords', [])}")

## 2. Data Overview
Let's review the available record sets, the fields within each set, and note their `@id`s for reference. This information is critical for selecting and loading specific parts of the dataset for further analysis.

In [ ]:
record_set_objs = list(dataset.metadata.record_sets)
print(f"Number of record sets: {len(record_set_objs)}\n")
for rset in record_set_objs:
    print(f"RecordSet name: {getattr(rset, 'name', None)}")
    print(f"  RecordSet @id: {rset.id}")
    field_ids = [f.id for f in rset.fields]
    print(f"  Fields: {field_ids}")
    print()

## 3. Data Extraction
Load data from a selected record set into a Pandas DataFrame for further analysis.

**NOTE:**
All entities are referenced using their `@id`. Edit `selected_record_set_id` if you wish to extract a different record set.

In [ ]:
# List all available RecordSet @ids
record_set_ids = [rset.id for rset in dataset.metadata.record_sets]

# Choose the main record set (in most Croissant tabular datasets the main set is usually the largest file)
selected_record_set_id = record_set_ids[0]
print(f"Selected RecordSet @id: {selected_record_set_id}\n")

# Load all records for each record set into a dictionary of DataFrames
dataframes = {}
for rset_id in record_set_ids:
    print(f"Loading records for RecordSet {rset_id} ...")
    recs = list(dataset.records(record_set=rset_id))
    df = pd.DataFrame(recs)
    print(f"  Loaded {len(df)} records and {len(df.columns)} columns.")
    dataframes[rset_id] = df

# Inspect the columns of the main DataFrame
main_df = dataframes[selected_record_set_id]
print("\nAvailable columns/fields (by @id):")
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Let's process the main DataFrame:
- Filter for records with high peptide counts (as an example numeric field)
- Normalize a numeric field (e.g., peptide count or coverage_pct)
- Optionally group by another field (e.g., a protein description/class)

**Please update the `numeric_field_id` and `group_field_id` in the code cell below to reference the correct Field `@id`s as shown above**.

In [ ]:
# Choose field @ids from the displayed columns above
# Example: If peptide count is cr:peptide_count and group by cr:description
numeric_field_id = None
group_field_id = None

# Try to auto-detect a numeric field with 'count' or 'coverage' in the column name
import numpy as np
for c in main_df.columns:
    if c.lower().endswith('count') or 'count' in c.lower():
        numeric_field_id = c
        break
if not numeric_field_id:
    # Try a default fallback
    for c in main_df.columns:
        if c.lower() in ['cr:peptide_count', 'cr:coverage_pct', 'cr:mw']:
            numeric_field_id = c
            break
if not group_field_id:
    for c in main_df.columns:
        if 'desc' in c.lower() or 'protein' in c.lower():
            group_field_id = c
            break

if numeric_field_id:
    print(f"Numeric field chosen: {numeric_field_id}")
    threshold = np.percentile(main_df[numeric_field_id].dropna(), 75)
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f} (top quartile):")
    print(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        print(f"\nGrouping by: {group_field_id}\n")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_value')
        print(grouped_df.head())
else:
    print("No suitable numeric field found. Please set numeric_field_id manually as shown above.")

## 5. Visualization
We'll visualize the numeric field distribution and compare values across groupings if possible.

You may customize these visualizations with any columns from your dataset as referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    # If available, show group-level means
    if group_field_id and group_field_id in main_df.columns:
        plt.figure(figsize=(10, 6))
        top_groups = main_df[group_field_id].value_counts().head(10).index
        filtered_plot_df = main_df[main_df[group_field_id].isin(top_groups)]
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_plot_df)
        plt.title(f'{numeric_field_id} by {group_field_id} (top 10)')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45, ha='right')
        plt.show()

## 6. Conclusion
With the `mlcroissant` library, we explored the FAIR² dataset's metadata and its main data. We filtered on, normalized, and visualized a key numeric field. For more targeted analyses, make sure to adjust the field IDs according to your domain objective.

**To go further:**
- Explore other record sets and fields using their `@id`s (as above).
- Join record sets if relational links are defined.
- Apply domain-specific validation and feature engineering.

For more information, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/) and the [dataset source](https://sen.science/doi/10.71728/senscience.vq4a-28xa).